# PCM: Simulate Traits for Comparative Inference

Trait simulation in phylogenetic comparative methods (PCM) is best viewed as an explicit *data-generating model* for evolutionary hypotheses. Rather than asking only whether an estimator can fit empirical data, simulation asks whether the estimator recovers known parameters under controlled conditions. This simulation-to-inference loop is central for method validation, power analysis, and conceptual understanding.

In this notebook, we focus on the user-facing simulation APIs in `toytree.pcm`:

- `toytree.pcm.simulate_continuous_trait`
- `toytree.pcm.simulate_discrete_trait`
- `toytree.pcm.simulate_multivariate_continuous_trait`
- `toytree.pcm.simulate_pgls_trait`
- `toytree.pcm.simulate_pglm_trait`


## Conceptual Background

### Continuous vs. discrete trait spaces

- **Continuous traits** (e.g., log body size) evolve on a numeric state space.
- **Discrete traits** (e.g., habitat classes) evolve on a finite set of categorical states.

These classes imply different stochastic processes, parameterizations, and inferential tools.

### Core model equations

For continuous traits, common models include:

- **Brownian motion (BM)**: random walk with branch-length-scaled variance.
  - Branch increment form: $\Delta X \sim \mathcal{N}(0, \sigma^2 t)$.
- **Ornstein-Uhlenbeck (OU)**: stochastic diffusion with attraction to an optimum.
  - SDE form: $dX_t = \alpha(\theta - X_t)dt + \sigma dW_t$.
- **Early burst (EB)**: time-varying evolutionary rate through exponential scaling.

For discrete traits, simulation follows a continuous-time Markov chain (CTMC):

- Instantaneous rate matrix $Q$ governs transitions among states.
- Branch transition probabilities are $P(t)=\exp(Qt)$.

Canonical references: Felsenstein (1985), Hansen (1997), Pagel (1994), Harmon (2019).


## Simulation Methods and Typical Pairings

| Simulator | Data Type | Typical Models | Typical Inference Pairing |
|---|---|---|---|
| `simulate_continuous_trait` | Univariate continuous | BM, OU, EB | Continuous trait model fitting, phylogenetic signal |
| `simulate_multivariate_continuous_trait` | Multivariate continuous | BM, OU, EB (matrix forms) | Multivariate covariance/PGLS-style workflows |
| `simulate_discrete_trait` | Discrete categorical | ER, SYM, ARD (CTMC) | Mk fitting and ancestral-state reconstruction |
| `simulate_pgls_trait` | Quantitative response | Formula + phylogenetic residual structure | PGLS fitting and parameter recovery checks |
| `simulate_pglm_trait` | Non-Gaussian response | Formula + GLM family/link + phylogeny | PGLM fitting and calibration checks |

Most simulation workflows are univariate by default. The multivariate continuous simulator is the main exception for correlated trait evolution.

In [1]:
import numpy as np

import toytree

tree = toytree.rtree.unittree(ntips=28, seed=123)

## Minimal Example (quick default usage)

Simulate one continuous trait under BM, store it directly on the tree (`inplace=True`), and plot it.

In [11]:
# simulate a continuous trait
tree.pcm.simulate_continuous_trait(
    model="bm",
    params=1.0,
    name="size",
    inplace=True,
    seed=10,
)

# plot the trait on a tree
tree.draw(node_colors=("size", "BlueRed"), node_sizes=10, node_mask=False, layout="d");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="563.6799999999998px" height="311.18px" viewBox="0 0 563.6799999999998 311.18" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="tf422b3ed69a0457ba5025b392dc431ac"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11 r12 r13 r14 r15 r16 r17 r18 r19 r20 r21 r22 r23 r24 r25 r26 r27

## Customization Example 1: Explicitly Parameterized Discrete Simulation

Here we move from defaults to a fully specified ARD CTMC with user-defined rates and equilibrium state frequencies.

In [13]:
rates = np.array(
    [
        [0.0, 2.0, 0.5],
        [1.0, 0.0, 1.2],
        [0.8, 0.3, 0.0],
    ]
)
freqs = np.array([0.5, 0.3, 0.2])

tree.pcm.simulate_discrete_trait(
    nstates=3,
    model="ARD",
    relative_rates=rates,
    state_frequencies=freqs,
    trait_name="state",
    state_names=["A", "B", "C"],
    inplace=True,
    seed=11,
)
c, a, m = tree.draw(
    layout="c",
    edge_type="p",
    node_colors=("state", "Set2"),
    node_sizes=8,
    node_mask=False,
)

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="500.0px" height="500.0px" viewBox="0 0 500.0 500.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="tb6a9c0a2645f43b4a94197910cac7f8e"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11 r12 r13 r14 r15 r16 r17 r18 r19 r20 r21 r22 r23 r24 r25 r26 r27

## Customization Example 2: Alternative Usage Pattern (`inplace=False`)

Return simulated data without storing to tree features. This is often useful in simulation studies where objects are assembled and transformed before plotting or fitting.

In [14]:
R = np.array([[1.0, 0.4], [0.4, 0.7]])
traits_mv = tree.pcm.simulate_multivariate_continuous_trait(
    model="bm",
    params=R,
    names=["trait1", "trait2"],
    tips_only=True,
    inplace=False,
    seed=12,
)
traits_mv.head()

,trait1,trait2
0,-0.359889,0.491545
1,0.428229,0.127817
2,0.200405,-0.411904
3,1.561195,0.812604
4,0.078625,0.001298


## Formula-Based Response Simulation: PGLS and PGLM

These simulators are especially useful when evaluating regression estimators under known phylogenetic dependence.

- `simulate_pgls_trait` produces a quantitative response under a Gaussian/phylogenetic structure.
- `simulate_pglm_trait` produces responses under GLM families (e.g., binomial, poisson) plus phylogenetic structure.

In [23]:
tree.pcm.simulate_continuous_trait(
    model="bm", params=0.8, name="x_cont", tips_only=True, inplace=True, seed=13
)
tree.pcm.simulate_discrete_trait(
    nstates=2, model="ER", trait_name="x_disc", tips_only=True, inplace=True, seed=14
)

y_pgls = tree.pcm.simulate_pgls_trait(
    formula="y ~ x_cont + x_disc",
    betas={"Intercept": 0.5, "x_cont": 1.2, "x_disc": -0.4},
    lambda_=0.7,
    sigma2=0.4,
    seed=15,
)

ToytreeError: betas keys must match Patsy design columns; missing beta keys: ['x_disc[T.1]']; unexpected beta keys: ['x_disc']

## Common Beginner Mistakes

- Mixing model names with incorrect parameter shapes (for example, OU/EB tuple parameters).
- Confusing tip-only returns with full node-indexed outputs.
- Forgetting whether simulated traits were written to tree features (`inplace=True`) or returned separately.
- In formula-based simulation, beta names must match Patsy-expanded design column names.

## Related APIs

- `tree.pcm.fit_discrete_ctmc(...)`
- `tree.pcm.infer_ancestral_states_discrete_ctmc(...)`
- `tree.pcm.fit_phylogenetic_gls(...)` (or package-level PGLS workflow)
- `tree.pcm.pglm(...)`
